# Train CNN-LSTM from 5° Pre-downloaded Data

Aligns MODIS (`paths.modis_5d`, default `data/modis_5d`) + S1 (`s1_labels_5d`), creates 64×64 patches, and trains the flood model.

**Requires:** MODIS GeoTIFFs and S1 `.npy` files under `data/modis_5d` and `data/s1_labels_5d` (or set `paths.modis_5d` in `config.yaml`).

**Training path (`training.data_mode` in `config.yaml`):** `legacy` — this notebook builds patches in RAM (cells 3–6). `tiles` — `Tile5dDataModule` from raw MODIS/S1. `patches` — `ShardedPatchDataModule` from `paths.processed_tiles_5d` (`patch_shard_*.npz`). With `loyo_cv.enabled: true`, cells 2 + 7 run leave-one-year-out for `tiles` or `patches` (patch LOYO needs `KEY` in each shard; re-run `process_5d_tiles.ipynb` if needed).

**Workflow (`training.data_mode` in `config.yaml`):** **`patches`** — pre-built `patch_shard_*.npz` only: run **cells 1–2**, then **cell 7** (FloodCNN_LSTM + Lightning), then **cell 8** (thesis figures). **Do not run cells 3–6** (in-notebook patch creation is for **`legacy`** only). Build shards with `process_5d_tiles.ipynb`. **`tiles`** — raw MODIS/S1 via `Tile5dDataModule`. **`legacy`** — RAM patches (cells 3–6). LOYO: set `loyo_cv.enabled: true`; patch LOYO needs `KEY` in each shard npz.


In [1]:
# Colab: mount Drive, install deps, set project path
from google.colab import drive
drive.mount('/content/drive')

!pip install -q rioxarray scipy scikit-learn pytorch-lightning pysheds
%cd /content/drive/MyDrive/indonesia-flood-cnn-lstm

# Data stays on the mounted Drive project folder (no copy to /content/data).
# Optional: set training.colab_copy_5d_to_local: true in config.yaml to copy 5deg tiles to SSD for data_mode=tiles only.


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/indonesia-flood-cnn-lstm


In [ ]:
# Setup: paths + optional Lightning datamodule (training.data_mode in config.yaml)
import os
import sys
from pathlib import Path

import numpy as np
import torch
import yaml

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

config = yaml.safe_load(open(PROJECT_ROOT / "config.yaml", encoding="utf-8")) if (PROJECT_ROOT / "config.yaml").exists() else {}
model_cfg = config.get("model", {})
train_cfg = config.get("training", {})
paths_cfg = config.get("paths", {})
loyo_cfg = config.get("loyo_cv", {})

DATA_MODE = str(train_cfg.get("data_mode", "legacy")).lower().strip()
_loyo_years = list(loyo_cfg.get("train_years", [2017, 2018, 2019, 2020, 2021]))
_LOYO_LOOP = bool(loyo_cfg.get("enabled", False)) and DATA_MODE in ("tiles", "patches")

_use_cuda = torch.cuda.is_available()

try:
    import google.colab  # noqa: F401

    _on_colab = True
except ImportError:
    _on_colab = False

if "num_workers" in train_cfg:
    _num_workers = int(train_cfg["num_workers"])
elif DATA_MODE == "tiles":
    _num_workers = 0
elif DATA_MODE == "patches":
    _num_workers = 0 if _on_colab else min(8, max(2, (os.cpu_count() or 8) // 2))
else:
    _num_workers = int(train_cfg.get("num_workers", 0))


def _path_on_drive(p: Path) -> bool:
    s = str(p.resolve()).lower()
    return "drive" in s or "my drive" in s


data_module = None

if DATA_MODE == "legacy":
    # Always use project data on the mounted Drive (no copy to /content/data).
    DATA_DIR = PROJECT_ROOT / "data"
    MODIS_DIR = DATA_DIR / "modis_5d"
    S1_DIR = DATA_DIR / "s1_labels_5d"
    print("data_mode=legacy — run cells 3-6 then cell 7 (in-memory patches + FloodDataModule)", flush=True)
    print(f"Project: {PROJECT_ROOT}", flush=True)
    print(f"MODIS: {MODIS_DIR} | S1: {S1_DIR}", flush=True)

elif DATA_MODE == "tiles":
    from pipeline.tile_5d_datamodule import Tile5dDataModule

    _copy_local = bool(train_cfg.get("colab_copy_5d_to_local", False)) and _on_colab
    _local_root = Path(train_cfg.get("colab_local_5d_root", "/content/indonesia_5d_local"))

    if _copy_local:
        from pipeline.colab_data_sync import ensure_local_5d_copy

        print("Colab: copying 5° MODIS + S1 to local SSD (tiles mode)...", flush=True)
        MODIS_DIR, S1_DIR = ensure_local_5d_copy(PROJECT_ROOT, _local_root, paths_cfg)
    else:
        MODIS_DIR = PROJECT_ROOT / (
            paths_cfg.get("modis_5d") or paths_cfg.get("modis_8day_5d") or "data/modis_5d"
        )
        S1_DIR = PROJECT_ROOT / paths_cfg.get("s1_labels_5d", "data/s1_labels_5d")

    _img_hw = int(model_cfg.get("img_hw", 64))
    _seq = int(model_cfg.get("sequence_length", 10))
    _max_tile = train_cfg.get("max_tile_keys")
    _max_tile = int(_max_tile) if _max_tile is not None else None
    _bs = int(train_cfg.get("tile_batch_size", train_cfg.get("batch_size", 8)))
    _mcache = int(train_cfg.get("tile5d_modis_cache_max", 48))
    _scache = int(train_cfg.get("tile5d_s1_cache_max", 96))

    print("data_mode=tiles — Tile5dDataModule (raw GeoTIFF + .npy)", flush=True)
    print("MODIS_DIR:", MODIS_DIR, "S1_DIR:", S1_DIR, flush=True)

    if _LOYO_LOOP:
        print("LOYO:", len(_loyo_years), "folds", _loyo_years, "— run cell 7", flush=True)
    else:
        data_module = Tile5dDataModule(
            modis_dir=MODIS_DIR,
            s1_dir=S1_DIR,
            img_hw=_img_hw,
            sequence_length=_seq,
            batch_size=_bs,
            num_workers=_num_workers,
            pin_memory=_use_cuda,
            val_split=float(train_cfg.get("val_split", 0.2)),
            random_state=42,
            max_tile_keys=_max_tile,
            modis_cache_max=_mcache,
            s1_cache_max=_scache,
        )
        print("--- data_module.setup('fit') ---", flush=True)
        data_module.setup("fit")
        print("--- done; run cell 7 ---", flush=True)

elif DATA_MODE == "patches":
    from pipeline.sharded_patch_datamodule import ShardedPatchDataModule

    PROC = PROJECT_ROOT / paths_cfg.get("processed_tiles_5d", "data/processed_tiles")
    manifest_path = PROC / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(
            "patches mode needs processed_tiles/manifest.json — run process_5d_tiles.ipynb first."
        )

    _max_open = int(
        train_cfg.get("sharded_max_open_shards", train_cfg.get("max_open_shards", 2 if _on_colab else 12))
    )
    _bs = int(train_cfg.get("batch_size", 32))
    _nw = 0 if _path_on_drive(PROC) else _num_workers
    if _path_on_drive(PROC):
        print("processed_tiles on Drive — DataLoader num_workers=0", flush=True)

    print(
        "data_mode=patches — ShardedPatchDataModule on pre-built patch_shard_*.npz "
        "(no in-notebook patch creation; run process_5d_tiles.ipynb to build shards).",
        flush=True,
    )
    print("PROC:", PROC, "max_open_shards=", _max_open, flush=True)

    if _LOYO_LOOP:
        print(
            "LOYO on shards needs KEY in each .npz — re-run process_5d_tiles.ipynb if shards are X/Y only.",
            flush=True,
        )
        print("LOYO:", len(_loyo_years), "folds", _loyo_years, "— run cell 7", flush=True)
    else:
        data_module = ShardedPatchDataModule(
            processed_tiles_dir=PROC,
            batch_size=_bs,
            num_workers=_nw,
            pin_memory=_use_cuda,
            val_split=float(train_cfg.get("val_split", 0.2)),
            random_state=42,
            max_open_shards=_max_open,
        )
        print("--- data_module.setup('fit') ---", flush=True)
        data_module.setup("fit")
        print("--- done; run cell 7 ---", flush=True)

else:
    raise ValueError(f"Unknown training.data_mode: {DATA_MODE!r} (use legacy | tiles | patches)")


data_mode=legacy — run cells 3-6 then cell 7 (in-memory patches + FloodDataModule)
Project: /content/drive/MyDrive/indonesia-flood-cnn-lstm
MODIS: /content/drive/MyDrive/indonesia-flood-cnn-lstm/data/modis_5d | S1: /content/drive/MyDrive/indonesia-flood-cnn-lstm/data/s1_labels_5d


In [ ]:
if DATA_MODE != "legacy":
    pass
else:
    if DATA_MODE != "legacy":
        pass
    else:
        if DATA_MODE != "legacy":
            pass
        else:
            if DATA_MODE != "legacy":
                pass
            else:
                # Scan available files
                def parse_modis_name(fname):
                    """Indo_5d_MODIS_2017_000_Tile_0.tif -> (2017, 0, 0)"""
                    stem = Path(fname).stem
                    if not stem.startswith('Indo_5d_MODIS_'): return None
                    parts = stem.replace('Indo_5d_MODIS_', '').split('_')
                    if len(parts) < 4: return None  # year, p_idx, 'Tile', tile_id
                    try:
                        return int(parts[0]), int(parts[1]), int(parts[-1])
                    except ValueError:
                        return None
    
                def parse_s1_name(fname):
                    """Indo_5d_S1_2017_000_Tile_0.npy -> (2017, 0, 0)"""
                    stem = Path(fname).stem
                    if not stem.startswith('Indo_5d_S1_'): return None
                    parts = stem.replace('Indo_5d_S1_', '').split('_')
                    if len(parts) < 4: return None
                    try:
                        return int(parts[0]), int(parts[1]), int(parts[-1])
                    except ValueError:
                        return None
    
                modis_files = {parse_modis_name(f.name): f for f in MODIS_DIR.glob('*.tif') if parse_modis_name(f.name)}
                s1_files = {parse_s1_name(f.name): f for f in S1_DIR.glob('*.npy') if parse_s1_name(f.name)}
    
                common_keys = set(modis_files.keys()) & set(s1_files.keys())
                print(f'MODIS files: {len(modis_files)}')
                print(f'S1 files: {len(s1_files)}')
                print(f'Common (year, p_idx, tile): {len(common_keys)}')

MODIS files: 5290
S1 files: 5283
Common (year, p_idx, tile): 5283


In [ ]:
if DATA_MODE != "legacy":
    pass
else:
    if DATA_MODE != "legacy":
        pass
    else:
        if DATA_MODE != "legacy":
            pass
        else:
            if DATA_MODE != "legacy":
                pass
            else:
                # Build MODIS time series (10 steps) + S1 label for each sample
                # Need periods p_idx-9 .. p_idx; handle year boundary
                from datetime import datetime, timedelta
                import rioxarray
                from scipy.ndimage import zoom
                import rasterio  # for RasterioIOError when skipping corrupted GeoTIFFs
    
                def get_8day_periods(year):
                    d = datetime(year, 1, 1)
                    n = 0
                    while d.year == year:
                        n += 1
                        d += timedelta(days=8)
                    return n
    
                SEQUENCE_LENGTH = 10
                PATCH_SIZE = 64
                PATCH_STRIDE = 64
                S1_PERCENTILE = 20  # Lower = water
    
                # Caches: avoid re-reading same files hundreds of times (major speedup)
                _modis_cache = {}
                _s1_cache = {}
    
                def load_sample(year, p_idx, tile_id):
                    """Load MODIS time series (10 steps) + S1 label. Returns (X, Y) or None."""
                    modis_list = []
                    for t in range(SEQUENCE_LENGTH):
                        pp = p_idx - (SEQUENCE_LENGTH - 1 - t)
                        yy = year
                        while pp < 0:
                            yy -= 1
                            pp += get_8day_periods(yy)
                        while pp >= get_8day_periods(yy):
                            pp -= get_8day_periods(yy)
                            yy += 1
                        key = (yy, pp, tile_id)
                        if key not in modis_files:
                            return None
                        if key not in _modis_cache:
                            try:
                                da = rioxarray.open_rasterio(modis_files[key])
                                _modis_cache[key] = np.asarray(da).squeeze()  # (7, H, W) or (H, W, 7)
                            except (rasterio.errors.RasterioIOError, OSError):
                                return None  # Corrupted GeoTIFF
                        modis_list.append(_modis_cache[key])
    
                    key = (year, p_idx, tile_id)
                    if key not in s1_files:
                        return None
                    if key not in _s1_cache:
                        _s1_cache[key] = np.load(s1_files[key])
                    s1 = _s1_cache[key]
    
                    # Stack MODIS: (10, 7, H, W)
                    modis_stack = np.stack(modis_list, axis=0)
                    if modis_stack.ndim == 4 and modis_stack.shape[1] == 7:
                        pass
                    elif modis_stack.ndim == 4 and modis_stack.shape[-1] == 7:
                        modis_stack = np.transpose(modis_stack, (0, 3, 1, 2))
                    else:
                        return None
    
                    # Align S1 to MODIS shape if needed
                    target_h, target_w = modis_stack.shape[2], modis_stack.shape[3]
                    if s1.shape != (target_h, target_w):
                        sy, sx = target_h / s1.shape[0], target_w / s1.shape[1]
                        s1 = zoom(s1, (sy, sx), order=1)
    
                    # S1 -> flood label (low backscatter = water)
                    valid = np.isfinite(s1) & (s1 > -999)
                    if valid.sum() < 100:
                        return None
                    thresh = np.percentile(s1[valid], S1_PERCENTILE)
                    flood = np.where(valid, (s1 < thresh).astype(np.float32), np.nan)
                    flood = np.nan_to_num(flood, nan=0.0)
    
                    # HAND placeholder (zeros) - 8th channel
                    hand = np.zeros((target_h, target_w), dtype=np.float32)
                    hand_expanded = np.repeat(hand[np.newaxis, np.newaxis, :, :], SEQUENCE_LENGTH, axis=0)
    
                    # X: (time, 8, H, W)
                    modis_norm = modis_stack / 10000.0
                    modis_norm = np.clip(np.nan_to_num(modis_norm, nan=0), 0, 1)
                    X = np.concatenate([modis_norm, hand_expanded], axis=1)
                    Y = flood
                    return X, Y

In [ ]:
if DATA_MODE != "legacy":
    pass
else:
    if DATA_MODE != "legacy":
        pass
    else:
        if DATA_MODE != "legacy":
            pass
        else:
            if DATA_MODE != "legacy":
                pass
            else:
                # Load all samples and create patches
                from pipeline.processor import create_patches
                from sklearn.model_selection import train_test_split
                from tqdm import tqdm
    
                MAX_SAMPLES = None  # Set to e.g. 500 for quick test; None = process all
                all_X = []
                all_Y = []
                n_skip = 0
                to_process = [k for k in sorted(common_keys) if k[1] >= SEQUENCE_LENGTH - 1]
                if MAX_SAMPLES:
                    to_process = to_process[:MAX_SAMPLES]
                    print(f"Limiting to {MAX_SAMPLES} samples (quick test mode)")
                for (year, p_idx, tile_id) in tqdm(to_process, desc="Loading samples"):
                    result = load_sample(year, p_idx, tile_id)
                    if result is None:
                        n_skip += 1
                        continue
                    X, Y = result
                    # Create patches
                    x_patches = create_patches(X, patch_size=PATCH_SIZE, stride=PATCH_STRIDE)  # (N, t, c, 64, 64)
                    y_patches = create_patches(Y, patch_size=PATCH_SIZE, stride=PATCH_STRIDE)  # (N, 64, 64)
                    if len(x_patches) == 0:
                        continue
                    all_X.append(x_patches)
                    all_Y.append(y_patches)
    
                X_all = np.concatenate(all_X, axis=0) if all_X else np.empty((0, 10, 8, 64, 64))
                Y_all = np.concatenate(all_Y, axis=0) if all_Y else np.empty((0, 64, 64))
                print(f'Skipped: {n_skip}')
                print(f'MODIS loaded (unique): {len(_modis_cache)}, S1 (unique): {len(_s1_cache)}')
                print(f'Total patches: {X_all.shape[0]}')
                print(f'X shape: {X_all.shape} (N, time, channels, 64, 64)')
                print(f'Y shape: {Y_all.shape}')

In [ ]:
if DATA_MODE != "legacy":
    pass
else:
    if DATA_MODE != "legacy":
        pass
    else:
        if DATA_MODE != "legacy":
            pass
        else:
            if DATA_MODE != "legacy":
                pass
            else:
                # Train/val split
                if X_all.shape[0] == 0:
                    raise ValueError("No patches created. Check that MODIS and S1 files exist and overlap.")
                X_train, X_val, Y_train, Y_val = train_test_split(X_all, Y_all, test_size=0.2, random_state=42, shuffle=True)
                print(f'Train: {X_train.shape[0]} patches')
                print(f'Val:   {X_val.shape[0]} patches')

In [ ]:
# Train: FloodCNN_LSTM + Lightning (run cell 2 first). Shards/patches: no cells 3–6.
import pytorch_lightning as pl
import torch
import yaml
from pathlib import Path
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

from pipeline.datamodule import FloodDataModule
from pipeline.model import FloodCNN_LSTM

config = yaml.safe_load(open(PROJECT_ROOT / "config.yaml", encoding="utf-8")) if (PROJECT_ROOT / "config.yaml").exists() else {}
model_cfg = config.get("model", {})
train_cfg = config.get("training", {})
paths_cfg = config.get("paths", {})

_logs = PROJECT_ROOT / "logs"
_logs.mkdir(exist_ok=True)

_precision = train_cfg.get("precision")
if _precision is None:
    _precision = "bf16-mixed" if torch.cuda.is_available() else "32-true"
else:
    _precision = str(_precision)

_max_epochs = int(train_cfg.get("max_epochs", 50))
_patience = int(train_cfg.get("patience", 10))
_loyo_epochs = int(train_cfg.get("epochs_per_loyo_fold", train_cfg.get("max_epochs", 5)))


def _make_model():
    return FloodCNN_LSTM(
        in_channels=int(model_cfg.get("in_channels", 8)),
        sequence_length=int(model_cfg.get("sequence_length", 10)),
        hidden_dim=int(model_cfg.get("hidden_dim", 128)),
        lr=float(model_cfg.get("learning_rate", 0.0001)),
        cnn_out_channels=int(model_cfg.get("cnn_out_channels", 64)),
        spatial_output_size=int(model_cfg.get("spatial_output_size", 4)),
    )


def _callbacks(subdir: str):
    d = _logs / subdir
    d.mkdir(parents=True, exist_ok=True)
    return [
        ModelCheckpoint(
            monitor="val_loss",
            dirpath=str(d),
            filename="best-{epoch:02d}-{val_loss:.4f}",
            save_top_k=1,
            mode="min",
        ),
        EarlyStopping(monitor="val_loss", patience=_patience, mode="min"),
    ]


def _csv_logger(name: str, version: str):
    return CSVLogger(save_dir=str(_logs), name=name, version=version)


if DATA_MODE == "legacy":
    dm = FloodDataModule(
        x_train=X_train,
        y_train=Y_train,
        x_val=X_val,
        y_val=Y_val,
        batch_size=int(train_cfg.get("batch_size", 32)),
        num_workers=int(train_cfg.get("num_workers", 0)),
    )
    model = _make_model()
    trainer = pl.Trainer(
        max_epochs=_max_epochs,
        accelerator="auto",
        precision=_precision,
        callbacks=_callbacks("legacy_ram"),
        logger=_csv_logger("legacy_ram", "run"),
        default_root_dir=str(_logs / "legacy_ram_pl"),
    )
    trainer.fit(model, datamodule=dm)

elif _LOYO_LOOP and DATA_MODE == "tiles":
    from pipeline.tile_5d_datamodule import Tile5dDataModule

    _img_hw = int(model_cfg.get("img_hw", 64))
    _seq = int(model_cfg.get("sequence_length", 10))
    _max_tile = train_cfg.get("max_tile_keys")
    _max_tile = int(_max_tile) if _max_tile is not None else None
    _bs = int(train_cfg.get("tile_batch_size", train_cfg.get("batch_size", 8)))
    _mcache = int(train_cfg.get("tile5d_modis_cache_max", 48))
    _scache = int(train_cfg.get("tile5d_s1_cache_max", 96))

    for val_year in _loyo_years:
        print("=== LOYO tiles val_year=", val_year, "===", flush=True)
        dm = Tile5dDataModule(
            modis_dir=MODIS_DIR,
            s1_dir=S1_DIR,
            img_hw=_img_hw,
            sequence_length=_seq,
            batch_size=_bs,
            num_workers=_num_workers,
            pin_memory=_use_cuda,
            val_split=float(train_cfg.get("val_split", 0.2)),
            random_state=42,
            max_tile_keys=_max_tile,
            modis_cache_max=_mcache,
            s1_cache_max=_scache,
            loyo_val_year=int(val_year),
            loyo_years_filter=_loyo_years,
        )
        dm.setup("fit")
        model = _make_model()
        trainer = pl.Trainer(
            max_epochs=_loyo_epochs,
            accelerator="auto",
            precision=_precision,
            callbacks=_callbacks(f"loyo_tiles_val{val_year}"),
            logger=_csv_logger("tiles_loyo", f"val_{val_year}"),
            default_root_dir=str(_logs / f"loyo_tiles_val{val_year}_pl"),
        )
        trainer.fit(model, dm)

elif _LOYO_LOOP and DATA_MODE == "patches":
    from pipeline.sharded_patch_datamodule import ShardedPatchDataModule

    PROC = PROJECT_ROOT / paths_cfg.get("processed_tiles_5d", "data/processed_tiles")
    _max_open = int(
        train_cfg.get("sharded_max_open_shards", train_cfg.get("max_open_shards", 4))
    )
    _bs = int(train_cfg.get("batch_size", 32))
    low = str(PROC.resolve()).lower()
    _nw = 0 if ("drive" in low or "my drive" in low) else int(train_cfg.get("num_workers", 0))

    for val_year in _loyo_years:
        print("=== LOYO patch shards val_year=", val_year, "===", flush=True)
        train_years = [y for y in _loyo_years if y != val_year]
        dm = ShardedPatchDataModule(
            processed_tiles_dir=PROC,
            batch_size=_bs,
            num_workers=_nw,
            pin_memory=_use_cuda,
            val_split=float(train_cfg.get("val_split", 0.2)),
            random_state=42,
            max_open_shards=_max_open,
            loyo_val_year=int(val_year),
            loyo_train_years=train_years,
        )
        dm.setup("fit")
        model = _make_model()
        trainer = pl.Trainer(
            max_epochs=_loyo_epochs,
            accelerator="auto",
            precision=_precision,
            callbacks=_callbacks(f"loyo_patches_val{val_year}"),
            logger=_csv_logger("patches_loyo", f"val_{val_year}"),
            default_root_dir=str(_logs / f"loyo_patches_val{val_year}_pl"),
        )
        trainer.fit(model, dm)

elif data_module is not None:
    model = _make_model()
    trainer = pl.Trainer(
        max_epochs=_max_epochs,
        accelerator="auto",
        precision=_precision,
        callbacks=_callbacks(str(DATA_MODE)),
        logger=_csv_logger(str(DATA_MODE), "run"),
        default_root_dir=str(_logs / f"{DATA_MODE}_pl"),
    )
    trainer.fit(model, data_module)

else:
    raise RuntimeError("No datamodule: check cell 2 and training.data_mode / loyo_cv.enabled.")


In [ ]:
# Thesis figures (after cell 7): bar chart + per-fold loss curves from Lightning CSV logs
import subprocess
import sys
from pathlib import Path

try:
    import matplotlib  # noqa: F401
    import pandas  # noqa: F401
    import seaborn  # noqa: F401
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "matplotlib", "seaborn", "pandas"]
    )

from pipeline.thesis_charts_5d import save_thesis_charts_from_lightning_logs

_thesis_meta = save_thesis_charts_from_lightning_logs(PROJECT_ROOT)
print("Thesis figures ->", _thesis_meta.get("figures_dir"))
print("summary.json ->", Path(_thesis_meta.get("figures_dir", "")) / "summary.json")
